# Module 11: Real-Time Object Tracking in Video
Unlike object *detection* (which searches an entire frame from scratch to find an object), object *tracking* takes a known target location from the first frame and follows its movement through subsequent frames. Tracking is computationally lighter and much faster because the system only looks in the immediate neighborhood of where the object was last seen. 


In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from zipfile import ZipFile
from urllib.request import urlretrieve
from IPython.display import HTML
from matplotlib.animation import FuncAnimation
from IPython.display import YouTubeVideo, display, HTML
from base64 import b64encode
%matplotlib inline

In [ ]:
def download_and_unzip(url, save_path):
    print(f"Downloading and extracting assests....", end="")

    # Downloading zip file using urllib package.
    urlretrieve(url, save_path)

    try:
        # Extracting zip file using the zipfile package.
        with ZipFile(save_path) as z:
            # Extract ZIP file contents in the same directory.
            z.extractall(os.path.split(save_path)[0])

        print("Done")

    except Exception as e:
        print("\nInvalid file.", e)

URL = r"https://www.dropbox.com/s/ld535c8e0vueq6x/opencv_bootcamp_assets_NB11.zip?dl=1"
asset_zip_path = os.path.join(os.getcwd(), f"opencv_bootcamp_assets_NB11.zip")

# Download if asset ZIP does not exist. 
if not os.path.exists(asset_zip_path):
    download_and_unzip(URL, asset_zip_path)

### 1. Baseline Video Verification
Before breaking the video into individual image frames for processing, we play the raw, unedited input file (`race_car.mp4`) here to verify that our local workspace paths and offline video configurations are fully functional.

In [ ]:
# Read the downloaded raw source file directly from your local workspace directory
mp4_initial = open("race_car.mp4", "rb").read()
data_url_initial = "data:video/mp4;base64," + b64encode(mp4_initial).decode()

# Display the initial unedited video stream element
HTML(f"""<video width=1024 controls><source src="{data_url_initial}" type="video/mp4"></video>""")

### 2. Choosing an Object Tracking Algorithm
OpenCV includes several tracking algorithms, each featuring different trade-offs between processing speed and tracking accuracy. In this module, we evaluate three primary types:

* **KCF (Kernelized Correlation Filters):** Extremely fast and efficient. It uses mathematical shortcuts to handle tracking patterns quickly but can lose track easily if the target changes shape or gets hidden behind another object.
* **CSRT (Channel and Spatial Reliability Tracker):** Highly accurate. It builds a more detailed mathematical map of the target's shape and boundary, making it great for complex movements, though it requires slightly more processing power.
* **GOTURN (Generic Object Tracking Using Regression Networks):** An AI deep-learning-based tracker. It uses a pre-trained neural network model to "predict" where the object moved based on thousands of reference video patterns.

In [ ]:
video_input_file_name = "race_car.mp4"

def drawRectangle(frame, bbox):
    p1 = (int(bbox[0]), int(bbox[1]))
    p2 = (int(bbox[0] + bbox[2]), int(bbox[1] + bbox[3]))
    cv2.rectangle(frame, p1, p2, (255, 0, 0), 2, 1)

def displayRectangle(frame, bbox):
    plt.figure(figsize=(20, 10))
    frameCopy = frame.copy()
    drawRectangle(frameCopy, bbox)
    frameCopy = cv2.cvtColor(frameCopy, cv2.COLOR_RGB2BGR)
    plt.imshow(frameCopy)
    plt.axis("off")

def drawText(frame, txt, location, color=(50, 170, 50)):
    cv2.putText(frame, txt, location, cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)

In [ ]:
tracker_types = [
    "BOOSTING",
    "MIL",
    "KCF",
    "CSRT",
    "TLD",
    "MEDIANFLOW",
    "GOTURN",
    "MOSSE",
]

# CHANGE THIS INDEX TO CHOOSE YOUR TRACKER:
# Index 1 = MIL (Built-in) | Index 2 = KCF | Index 3 = CSRT | Index 6 = GOTURN (Built-in)
tracker_type = tracker_types[6] 

# Helper function to safely find and build tracker objects across different OpenCV versions
def create_tracker(name):
    # 1. Check for MIL (always available)
    if name == "MIL":
        if hasattr(cv2, "TrackerMIL_create"): return cv2.TrackerMIL_create()
        if hasattr(cv2, "TrackerMIL"): return cv2.TrackerMIL.create()
        
    # 2. Check for GOTURN (always available)
    elif name == "GOTURN":
        if hasattr(cv2, "TrackerGOTURN_create"): return cv2.TrackerGOTURN_create()
        if hasattr(cv2, "TrackerGOTURN"): return cv2.TrackerGOTURN.create()
        
    # 3. Check for KCF (requires contrib modules)
    elif name == "KCF":
        if hasattr(cv2, "TrackerKCF_create"): return cv2.TrackerKCF_create()
        if hasattr(cv2, "TrackerKCF"): return cv2.TrackerKCF.create()
        
    # 4. Check for CSRT (requires contrib modules)
    elif name == "CSRT":
        if hasattr(cv2, "TrackerCSRT_create"): return cv2.TrackerCSRT_create()
        if hasattr(cv2, "TrackerCSRT"): return cv2.TrackerCSRT.create()
        
    # If the code reaches here, the requested tracker is missing from this active environment
    return None

# Attempt to build the selected tracker
tracker = create_tracker(tracker_type)

if tracker is not None:
    print(f"Successfully initialized the {tracker_type} tracker module object!")
else:
    print("\n" + "="*80)
    print(f" ENVIRONMENT NOTICE: '{tracker_type}' cannot be loaded by your active VS Code kernel.")
    print(" -> To run immediately: Change your tracker index above to 1 (MIL) or 6 (GOTURN).")
    print(" -> To fix KCF/CSRT: Click the 'Kernel' button in the top-right of VS Code, ")
    print("    select 'Restart Kernel', and make sure 'opencv-env' is chosen.")
    print("="*80 + "\n")
    raise RuntimeError(f"Could not build the {tracker_type} tracker structure.")

### 3. Setting Up the Video Pipeline
We initialize our input video stream using `cv2.VideoCapture`. Once open, we extract the frame's exact pixel width and height so that our output `cv2.VideoWriter` can generate a matching output file compressed using the standard XVID format.

In [ ]:
# Read video
video = cv2.VideoCapture(video_input_file_name)
ok, frame = video.read()

# Exit if video not opened
if not video.isOpened():
    print("Could not open video")
    sys.exit()
else:
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

video_output_file_name = "race_car-" + tracker_type + ".mp4"
video_out = cv2.VideoWriter(video_output_file_name, cv2.VideoWriter_fourcc(*"XVID"), 10, (width, height))

### 4. Setting the Tracking Target
To start the tracker, we must feed it a starting coordinate box known as a **Bounding Box (Bounding Box / ROI)**. The array format represents `(X_start, Y_start, Width, Height)`. This box acts as the initial "search template" for the car we want to follow.

In [ ]:
# Define a bounding box tracking target
bbox = (1300, 405, 160, 120)
# bbox = cv2.selectROI(frame, False)
# print(bbox)
displayRectangle(frame, bbox)

In [ ]:
# Initialize tracker with first frame and bounding box coordinates
ok = tracker.init(frame, bbox)

### 5. The Frame Processing & Tracking Loop
This loop reads the video frame-by-frame. 
* **The Math:** For each new frame, `tracker.update(frame)` runs to compute where the target has moved. 
* **Performance Tracking:** We track execution time using `cv2.getTickCount()` to calculate the **Frames Per Second (FPS)**. This tells us exactly how fast the tracking math is executing on our local hardware.
* **Failure Prevention:** If the car moves off-screen or goes behind an obstacle, the tracker will return `False`, prompting a "Tracking failure detected" warning on the frame rather than crashing the script.

In [ ]:
while True:
    ok, frame = video.read()

    if not ok:
        break

    # Start tracking stopwatch timer
    timer = cv2.getTickCount()

    # Update tracker bounding box estimations
    ok, bbox = tracker.update(frame)

    # Calculate Frames per second (FPS)
    fps = cv2.getTickFrequency() / (cv2.getTickCount() - timer)

    # Draw bounding box on active frame matrices
    if ok:
        drawRectangle(frame, bbox)
    else:
        drawText(frame, "Tracking failure detected", (80, 140), (0, 0, 255))

    # Display live information text overlays
    drawText(frame, tracker_type + " Tracker", (80, 60))
    drawText(frame, "FPS : " + str(int(fps)), (80, 100))

    # Write frame matrix data to file
    video_out.write(frame)

# Release resources cleanly
video.release()
video_out.release()
print("Tracking frame matrices calculated and saved.")

In [ ]:
# Change video encoding of compiled tracker output file from XVID to web-safe h264
# Using a unique name template so the different tracker results do not overwrite each other
video_output_encoded = f"race_car_track_{tracker_type}_x264.mp4"
!ffmpeg -y -i {video_output_file_name} -c:v libx264 {video_output_encoded} -hide_banner -loglevel error
print(f"Output tracking stream successfully transcoded to {video_output_encoded}.")

### 6. Side-by-Side Performance Evaluation
Below are the final compiled tracking outputs for our three evaluated tracking modules. By looking at them one under the other, we can directly compare their execution speeds (FPS counters) and tracking stability under sudden camera movements.

In [ ]:
# Read and play the finished KCF tracking video
mp4_kcf = open("race_car_track_KCF_x264.mp4", "rb").read()
data_url_kcf = "data:video/mp4;base64," + b64encode(mp4_kcf).decode()

print("Displaying Output for: KCF Tracker")
HTML(f"""<video width=1024 controls><source src="{data_url_kcf}" type="video/mp4"></video>""")

In [ ]:
# Read and play the finished CSRT tracking video
mp4_csrt = open("race_car_track_CSRT_x264.mp4", "rb").read()
data_url_csrt = "data:video/mp4;base64," + b64encode(mp4_csrt).decode()

print("Displaying Output for: CSRT Tracker")
HTML(f"""<video width=1024 controls><source src="{data_url_csrt}" type="video/mp4"></video>""")

In [ ]:
# Read and play the finished GOTURN tracking video
mp4_goturn = open("race_car_track_GOTURN_x264.mp4", "rb").read()
data_url_goturn = "data:video/mp4;base64," + b64encode(mp4_goturn).decode()

print("Displaying Output for: GOTURN Tracker")
HTML(f"""<video width=1024 controls><source src="{data_url_goturn}" type="video/mp4"></video>""")